In [1]:
import json, time, os
import requests
import pandas as pd

from enum import Enum
from typing import Any, List, Dict, Optional, Union

SCRYFALL_BASE_URL = "https://api.scryfall.com/"
os.getcwd()

'/Users/ptallo/Documents/mtg_data_science'

# Utils

In [10]:
def write_file(fpath: str, content: Union[str, bytes]) -> None:
    if not os.path.exists(os.path.dirname(fpath)):
        os.makedirs(os.path.dirname(fpath))
    if isinstance(content, str):
        with open(fpath, "wb") as f:
            f.write(content.encode("utf-8"))
    elif isinstance(content, bytes):
        with open(fpath, "wb") as f:
            f.write(content)
    
def read_file(fpath: str, encoding: str = "utf-8") -> str:
    with open(fpath, "r", encoding=encoding) as f:
        return f.read()

# Caching and Requesting

In [11]:
class CacheType(Enum):
    JSON = "json"
    PNG = "png"

class CacheHandler:
    def __init__(self, cache_dir: str, cache_type: CacheType = CacheType.JSON):
        self.cache_dir = cache_dir
        self.cache_type = cache_type

    def _get_key_fpath(self, key: str) -> str:
        return os.path.join(self.cache_dir, f'{key}.{self.cache_type.value}')

    def get(self, key: str) -> Optional[str]:
        fpath = self._get_key_fpath(key)
        return read_file(fpath) if os.path.exists(fpath) else None

    def set(self, key: str, value: Union[str, bytes]) -> None:
        fpath = self._get_key_fpath(key)
        write_file(fpath, value)

    def delete(self, key: str) -> None:
        fpath = self._get_key_fpath(key)
        if not os.path.exists(fpath):
            return
        os.remove(fpath)

    def clear(self) -> None:
        files = [os.path.join(self.cache_dir, f) for f in os.listdir(self.cache_dir)]
        [self.delete(f) for f in files]

In [17]:
class ScryfallHttpClient:
    def _get_scryfall_headers(self) -> Dict[str, str]:
        return {
            "User-Agent": "MTG-DataScience-Project/1.0",
            "Accept": "application/json;q=0.9,*/*,q=0.8",
        }
    
    def get(self, url: str, **kwargs) -> requests.Response:
        response = requests.get(url, headers=self._get_scryfall_headers(), **kwargs)
        if response.status_code != 200:
            response.raise_for_status()
        return response

In [15]:
class ScryfallClient:
    def __init__(self, base_url: str, cache_handler: CacheHandler, http_client: ScryfallHttpClient):
        self.cache_handler = cache_handler
        self.http_client = http_client
        self.base_url = base_url
        self.endpoints = {
            "bulk-data": "bulk-data",
        }

    def get_bulk_data(self) -> pd.DataFrame:
        cache_contents = self.cache_handler.get(self.endpoints['bulk-data'])
        if cache_contents is not None:
            return pd.DataFrame(json.loads(cache_contents).get("data"))

        response_json = self.http_client.get(f"{self.base_url}/{self.endpoints['bulk-data']}")
        self.cache_handler.set(self.endpoints['bulk-data'], json.dumps(response_json.json(), indent=4))
        cache_contents = self.cache_handler.get(self.endpoints['bulk-data'])
        return pd.DataFrame(json.loads(cache_contents).get("data"))

    def get_oracle_cards(self) -> pd.DataFrame:
        cache_key = 'oracle-cards'
        res_df = self.get_bulk_data()
        download_uri = res_df[res_df['name'] == 'Oracle Cards']['download_uri'].iloc[0]

        cache_contents = self.cache_handler.get(cache_key)
        if cache_contents is not None:
            return pd.DataFrame(json.loads(cache_contents))

        response = self.http_client.get(download_uri, stream=True)
        self.cache_handler.set(cache_key, response.content)
        cache_contents = self.cache_handler.get(cache_key) 
        return pd.DataFrame(json.loads(cache_contents))

# Fetching Images

In [ ]:
json_cache_handler = CacheHandler("./cache/json/", cache_type=CacheType.JSON)
scryfall_http_client = ScryfallHttpClient()
sfc = ScryfallClient(SCRYFALL_BASE_URL, json_cache_handler, scryfall_http_client)

Index(['object', 'id', 'oracle_id', 'multiverse_ids', 'mtgo_id',
       'tcgplayer_id', 'cardmarket_id', 'name', 'lang', 'released_at', 'uri',
       'scryfall_uri', 'layout', 'highres_image', 'image_status',
       'image_updated_at', 'image_uris', 'mana_cost', 'cmc', 'type_line',
       'oracle_text', 'power', 'toughness', 'colors', 'color_identity',
       'keywords', 'all_parts', 'legalities', 'games', 'reserved',
       'game_changer', 'foil', 'nonfoil', 'finishes', 'oversized', 'promo',
       'reprint', 'variation', 'set_id', 'set', 'set_name', 'set_type',
       'set_uri', 'set_search_uri', 'scryfall_set_uri', 'rulings_uri',
       'prints_search_uri', 'collector_number', 'digital', 'rarity',
       'watermark', 'flavor_text', 'card_back_id', 'artist', 'artist_ids',
       'illustration_id', 'border_color', 'frame', 'frame_effects',
       'security_stamp', 'full_art', 'textless', 'booster', 'story_spotlight',
       'edhrec_rank', 'preview', 'prices', 'related_uris', 'purchase

In [28]:
oracle_df = sfc.get_oracle_cards()
oracle_df.shape

(38233, 88)

In [29]:
commanders_df = pd.DataFrame(read_file("./data/commanders.txt").split("\n"), columns=["name"])
commanders_df.shape

(28, 1)

In [30]:
merged_df = commanders_df.merge(oracle_df[['name', 'oracle_text']], on="name", how="inner")
merged_df.shape

(27, 2)